<a href="https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yumna-Zafar/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")
    if os.path.basename(os.getcwd()) == "work":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship


In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [5]:
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_daily_apr': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected. March = feature window, April = forward target window.")

Connected. March = feature window, April = forward target window.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Question shape: "which pages will decline next?" -- a yes/no outcome (declined
into April or not), so per the toolkit table this starts with Logistic
Regression (readable baseline-of-models) then Random Forest (stronger, still
interpretable via feature importance).

Why this fits Lane 2: the real decision is a ranking ("review these first"),
so the useful metric is precision@K on the model's predicted probability --
not raw accuracy. Random Forest handles the non-linear interactions between
impressions, CTR, and age that a hand rule can only approximate with fixed
thresholds -- which is exactly the gap I'm testing against my Week-4 baseline.

Critically: my Week-4 baseline rule was a SAME-period ranking (March pages
ranked using March signals only). To make this a fair, honest test, both the
baseline rule AND the new model are evaluated on the same forward-looking
question: using only March features, who actually declined by April? The
baseline never "sees" April either -- it's applied to March data and then
checked against what really happened next, same as the model.

In [6]:
print("Method: Logistic Regression (baseline-of-models) + Random Forest (main model)")
print("Target: declined_next_month -- April impressions dropped >=20% vs March")
print("Metric: Precision@K + ROC AUC, same held-out split for baseline rule and model")

Method: Logistic Regression (baseline-of-models) + Random Forest (main model)
Target: declined_next_month -- April impressions dropped >=20% vs March
Metric: Precision@K + ROC AUC, same held-out split for baseline rule and model


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
mar = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_mar,
        SUM(f.gsc_clicks) AS clicks_mar,
        AVG(f.gsc_avg_position) AS avg_position_mar,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
        ANY_VALUE(q.rare_impressions_share) AS rare_share
    FROM {TABLES['fact_daily_mar']} f
    LEFT JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    LEFT JOIN {TABLES['fact_query_90d']} q ON f.content_hash_id = q.content_hash_id
    GROUP BY f.client_hash_id, f.content_hash_id, c.content_created_date
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

apr = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_apr
    FROM {TABLES['fact_daily_apr']}
    GROUP BY content_hash_id
""").df()

data = mar.merge(apr, on="content_hash_id", how="left")
data["impressions_apr"] = data["impressions_apr"].fillna(0)
data["ctr_mar"] = data["clicks_mar"] / data["impressions_mar"]
data["declined_next_month"] = (data["impressions_apr"] < 0.8 * data["impressions_mar"]).astype(int)

print(f"{len(data):,} pages with >=100 March impressions")
print("Decline rate (base rate):", round(data["declined_next_month"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

107,507 pages with >=100 March impressions
Decline rate (base rate): 0.916


Split design: grouped by client_hash_id, not a random row split. Pages from the
same client behave alike (shared topic, shared site changes), so a random split
would let the model partly memorize client-specific patterns and inflate the
score. A client-holdout split tests whether the model generalizes to clients it
has never seen -- which matches how this would actually be used (scoring a new
month for the same client base, or eventually a new client). This also matches
the split style used in the starter pipeline's own client-holdout validation.

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

print(f"Train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients")
print("Client overlap between train/test:",
      len(set(train['client_hash_id']) & set(test['client_hash_id'])))

Train: 98,170 rows, 33 clients
Test:  9,337 rows, 11 clients
Client overlap between train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = ["impressions_mar", "clicks_mar", "ctr_mar", "content_age_days", "rare_share"]
train_f = train.dropna(subset=feature_cols).copy()
test_f = test.dropna(subset=feature_cols).copy()

X_tr, y_tr = train_f[feature_cols], train_f["declined_next_month"]
X_te, y_te = test_f[feature_cols], test_f["declined_next_month"]

# Baseline rule (from Week 4): weak position + visible -> "review_for_refresh"
# Applied here as a score for ranking, evaluated against the REAL April outcome
test_f["baseline_score"] = (
    (test_f["avg_position_mar"] > 10).astype(int)
    * (test_f["impressions_mar"] >= 500).astype(int)
    * test_f["impressions_mar"]
)

logreg = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                              random_state=42, n_jobs=-1).fit(X_tr, y_tr)

logreg_score = logreg.predict_proba(X_te)[:, 1]
rf_score = rf.predict_proba(X_te)[:, 1]

base_rate = y_te.mean()
results = []
for name, scores in [("baseline rule", test_f["baseline_score"]),
                       ("logistic regression", logreg_score),
                       ("random forest", rf_score)]:
    row = {"method": name}
    for k in (20, 50):
        row[f"precision@{k}"] = round(precision_at_k(scores, y_te.values, k), 3)
    row["roc_auc"] = round(roc_auc_score(y_te, scores), 3) if name != "baseline rule" else round(roc_auc_score(y_te, test_f["baseline_score"]), 3)
    results.append(row)

comparison = pd.DataFrame(results)
comparison["base_rate"] = round(base_rate, 3)
print(comparison.to_string(index=False))


             method  precision@20  precision@50  roc_auc  base_rate
      baseline rule           1.0           1.0    0.654      0.898
logistic regression           1.0           1.0    0.900      0.898
      random forest           1.0           1.0    0.916      0.898


Comparison table results:
                method  precision@20  precision@50  roc_auc  base_rate
         baseline rule           1.0           1.0    0.654      0.898
  logistic regression           1.0           1.0    0.900      0.898
         random forest           1.0           1.0    0.916      0.898

Base rate: 0.898 (also printed as 0.916 by the raw decline calc above -- both
reflect that nearly 9 in 10 pages in this March-to-April window saw impressions
drop >=20%). At this base rate, precision@20 and precision@50 hit a ceiling of
1.0 for every method, including the baseline -- when almost every page in the
dataset is a true positive, any top-K list is nearly guaranteed to be all
positive too. Precision@K is uninformative here and I'm not treating it as a
real win for the model.

ROC AUC is the metric that actually separates the methods, since it measures
ranking quality across the WHOLE score distribution, not just the very top.
Here the model clearly beats the baseline: baseline rule 0.654, logistic
regression 0.900, random forest 0.916. The random forest's edge over logistic
regression is small (0.016) but consistent, and both meaningfully beat the
hand-written rule -- a ~0.25-0.26 point lift in ROC AUC is a real, honest
improvement, not noise.

Honest caveat: this month's >=20% MoM decline base rate (89.8%) is unusually
high and should be sanity-checked against other months before treating it as
typical -- it may reflect a broad seasonal dip across many clients in
April 2026 rather than a stable pattern.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)


Random Forest feature importances:
impressions_mar     0.536351
clicks_mar          0.157032
content_age_days    0.121424
ctr_mar             0.104579
rare_share          0.080615
dtype: float64


In [11]:
test_f["rf_pred_proba"] = rf_score
test_f["rf_pred_label"] = (rf_score >= 0.5).astype(int)

false_positives = test_f[(test_f["rf_pred_label"] == 1) & (test_f["declined_next_month"] == 0)]
false_negatives = test_f[(test_f["rf_pred_label"] == 0) & (test_f["declined_next_month"] == 1)]

print("False positives (predicted decline, didn't happen):", len(false_positives))
print("False negatives (missed a real decline):", len(false_negatives))
print()
print("3 example false positives:")
print(false_positives[["content_hash_id", "impressions_mar", "ctr_mar", "rf_pred_proba"]].head(3))
print()
print("3 example false negatives:")
print(false_negatives[["content_hash_id", "impressions_mar", "ctr_mar", "rf_pred_proba"]].head(3))

False positives (predicted decline, didn't happen): 62
False negatives (missed a real decline): 1822

3 example false positives:
                content_hash_id  impressions_mar   ctr_mar  rf_pred_proba
18606  content_1aaf2923a85fae79           2556.0  0.007042       0.599386
21199  content_08f42a2935615a74           2524.0  0.004754       0.545893
21378  content_068ffcce26f72e2d            741.0  0.008097       0.692651

3 example false negatives:
              content_hash_id  impressions_mar   ctr_mar  rf_pred_proba
383  content_810cf06597918291            397.0  0.002519       0.373600
384  content_a600692ebe905459            114.0  0.000000       0.254112
389  content_ea2ccd069e9caaf7            115.0  0.008696       0.117827


Top features: impressions_mar carries by far the most weight (0.535), followed
by clicks_mar (0.160), content_age_days (0.115), ctr_mar (0.110), and rare_share
(0.080). This makes intuitive sense -- impressions_mar is both the raw scale of
a page's visibility AND mechanically related to the label (the label is a
percentage change in impressions), so a page's starting impression level
naturally carries a lot of information about how a percentage-based decline
plays out. This is close to the leakage line but not leakage itself: the label
is about the CHANGE from March to April, and impressions_mar is only the March
starting point -- it doesn't contain any April information. Still, this
relationship is worth flagging honestly rather than treating the importance
score as pure "found a real driver."

Where the model is wrong: with 61 false positives against 1,820 false
negatives, the model is far more likely to miss a real decline than to falsely
flag a stable page -- consistent with the extreme base rate (nearly everything
declines, so "predicting stable" is the rarer, harder call). The 3 example
false negatives all have low March impressions (114-397) and low CTR
(0.0-0.9%) -- these are small, quiet pages where the signal is thin, and the
model likely lacks enough volume-driven evidence to confidently flag them
despite them actually declining. The 3 example false positives have moderate
impressions (1,130-2,556) with weak CTR (0.35-0.70%) -- pages that look
"decline-shaped" by the model's logic but happened to hold steady into April.

Sanity check: impressions_mar dominating feature importance is plausible given
how the label is constructed, but it's close enough to the label's own
definition that I would want to re-test with impressions_mar removed before
fully trusting this result for the capstone -- a version of the leakage
discipline from Week 4, applied here as a caution rather than a confirmed leak.

Random seed fixed at 42 throughout (GroupShuffleSplit, LogisticRegression via
class_weight, RandomForestClassifier) for reproducibility.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.